In [62]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNet
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [63]:
def extract_series(df):
    df_temp = df

    # Convert 'id' column to string type
    df_temp['id'] = df_temp['id'].astype(str)
    # Add a new column 'series' which is the first three digits of 'id'
    df_temp['series'] = df_temp['id'].str[:4].astype(int)

    # Read the series.csv file
    series = pd.read_csv('Series Name.csv')

    # Merge data with series on the 'series' column from data and 'series_id' column from series
    df_temp_1 = df_temp.merge(series, left_on='series', right_on='Series Name ID', how='left')

    cols = df_temp_1.columns.tolist()
    cols.insert(1, cols.pop(cols.index('Series Name')))
    df_temp_1 = df_temp_1[cols]
    
    df_temp_1.drop(columns=['Series Name ID', 'series'], inplace=True)

    return df_temp_1

In [64]:
def extract_contry(df):
    df_temp = df
    # Add a new column 'country' which is the fourth and fifth digits of 'id'
    df_temp['country'] = df_temp['id'].str[4:6].astype(int)

# Read the country.csv file
    country = pd.read_csv('Country Name.csv')

# Merge data_series with country on the 'country' column from data_series and 'country_id' column from country
    df_temp_1 = df_temp.merge(country, left_on='country', right_on='Country Name ID', how='left')

# Insert 'location' column into the correct position
    cols = df_temp_1.columns.tolist()
    cols.insert(2, cols.pop(cols.index('Country Name')))
    df_temp_1 = df_temp_1[cols]

# Drop unnecessary columns
    df_temp_1.drop(columns=['Country Name ID', 'country'], inplace=True)
    
    return df_temp_1

In [65]:
def extract_category(df):
    df_temp = df
    # Add a new column 'country' which is the fourth and fifth digits of 'id'
    df_temp['category'] = df_temp['id'].str[6:8].astype(int)

# Read the country.csv file
    category = pd.read_excel('category_id.xlsx')

# Rename the 'id' column in the country dataframe to 'country_id'
    category.rename(columns={'id': 'category_id'}, inplace=True)

# Merge data_series with country on the 'country' column from data_series and 'country_id' column from country
    df_temp_1 = df_temp.merge(category, left_on='category', right_on='category_id', how='left')

# Insert 'location' column into the correct position
    cols = df_temp_1.columns.tolist()
    cols.insert(3, cols.pop(cols.index('Category')))
    df_temp_1 = df_temp_1[cols]

# Drop unnecessary columns
    df_temp_1.drop(columns=['category_id', 'category'], inplace=True)

    return df_temp_1

In [66]:
prosperity = pd.read_csv('prosperity.csv')
prosperity.columns = ['year', 'prosperity']

In [67]:
data = pd.read_csv('cleaned_data.csv')
data['id'] = data['id'].astype(str)
data['id'] = data['id'].apply(lambda x: f'{int(x):014}' if pd.notnull(x) else x)
#nf_data = data[data['id'].str[5:7] != '05']
data = data.set_index('id')
data_stand = data.copy()

In [68]:
for col in range(len(data)):
    data_stand.iloc[col] = (data.iloc[col] - data.iloc[col].min()) / (data.iloc[col].max() - data.iloc[col].min())
data_stand = data_stand.dropna()

In [69]:
# Create a dataframe to store the id and p-values
pvalues_df = pd.DataFrame(columns=['id', 'pvalue'])

for col in range(len(data_stand)):

    # Ensure y and x have the same length
    x = data_stand.iloc[col].values.reshape(-1, 1)
    y = prosperity['prosperity']

    # Add a constant to the model (intercept)
    x = sm.add_constant(x)
    model = sm.OLS(y, x).fit()

    pvalues_df.loc[col, 'id'] = data.index[col]
    
    # Check if model.pvalues has an index 1
    if len(model.pvalues) > 1:
        pvalues_df.loc[col, 'pvalue'] = model.pvalues[1]  # Extract the p-value for the predictor variable
    else:
        pvalues_df.loc[col, 'pvalue'] = 1 # Assign NaN if p-value is not available


/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_25662/2330189477.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pvalues_df.loc[col, 'pvalue'] = model.pvalues[1]  # Extract the p-value for the predictor variable
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_25662/2330189477.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pvalues_df.loc[col, 'pvalue'] = model.pvalues[1]  # Extract the p-value for the predictor variable
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_25662/2330189477.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a futu

In [70]:
pvalues_threshold = 0.01 / len(pvalues_df)
data_significant = pvalues_df[pvalues_df['pvalue'] < pvalues_threshold]

In [71]:
# Convert 'pvalue' column to numeric
pvalues_df['pvalue'] = pd.to_numeric(pvalues_df['pvalue'], errors='coerce')


In [72]:
temp = data_stand.reset_index()

# Get the top 10 rows with the smallest p-values
#nf_data_top10_pvalues = pvalues_df.nsmallest(100, 'pvalue')
#nf_data_top10_pvalues = nf_data_significant

data_significant_db = temp[temp['id'].isin(data_significant['id'])]

data_db = data_significant_db

In [73]:
data_significant_seires = extract_series(data_significant_db)
data_significant_country = extract_contry(data_significant_seires)
data_significant_category = extract_category(data_significant_country)

/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_25662/395400177.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['id'] = df_temp['id'].astype(str)
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_25662/395400177.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['series'] = df_temp['id'].str[:4].astype(int)


In [74]:
len(data_significant_category)

394

In [75]:
data_significant_category.to_excel('pvalue_data.xlsx')

In [76]:
data_significant_category.groupby('Series Name').count()['id'].sort_values(ascending=False).to_csv('significant.csv')

In [77]:
data_db = data_db.drop('series', axis=1)

In [78]:
data_db_clean = data_db.set_index('id').transpose()

prosperity.index = data_db_clean.index

prosperity_clean = prosperity.drop(columns=['year'])

In [88]:
def compute_vif(X):
    if X.empty:
        return pd.DataFrame(columns=["variable", "VIF"])
    vif = pd.DataFrame()
    vif["variable"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif


In [ ]:

def stepwise_selection(X, y, threshold_in=0.01, vif_threshold=10, verbose=True):
    included = []
    while True:
        changed = False
        
        # Forward Step: Evaluate adding candidates not already included.
        excluded = list(set(X.columns) - set(included))
        new_pvals = pd.Series(index=excluded, dtype=float)
        for new_var in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [new_var]])).fit(cov_type='HC3')
            new_pvals[new_var] = model.pvalues[new_var]
        if not new_pvals.empty:
            best_pval = new_pvals.min()
            if best_pval < threshold_in:
                best_var = new_pvals.idxmin()
                included.append(best_var)
                changed = True
                if verbose:
                    print(f"Add {best_var:30} with p-value {best_pval:.6f}")
                    
                # Check VIF after adding the new variable
                if included:  # Ensure included list is not empty
                    current_vif = compute_vif(X[included])
                    if not current_vif.empty and not current_vif['VIF'].isnull().all() and current_vif['VIF'].max() > vif_threshold:
                        if verbose:
                            print(f"Warning: High VIF detected. Current VIFs:\n{current_vif}")
                        # Optionally, remove the variable with the highest VIF
                        worst_vif_var = current_vif.sort_values("VIF", ascending=False).iloc[0]['variable']
                        included.remove(worst_vif_var)
                        if verbose:
                            print(f"Removed {worst_vif_var} due to high VIF.")
        
        if not changed:
            break
    
    return included
selected_vars = stepwise_selection(data_db_clean, prosperity_clean)
print("Final selected predictors:", selected_vars)


Add 06040104000604                 with p-value 0.000000


ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:

# Create a DataFrame to store the id, p-value, and coefficient
results_df = pd.DataFrame(columns=['id', 'pvalue', 'coefficient'])

for var in selected_vars:
    # Extract the p-value and coefficient for each selected variable
    model = sm.OLS(prosperity['prosperity'], sm.add_constant(data_db_clean[var])).fit()
    pvalue = model.pvalues[var]
    coefficient = model.params[var]
    
    # Append the results to the DataFrame
    results_df = results_df.append({'id': var, 'pvalue': pvalue, 'coefficient': coefficient}, ignore_index=True)


In [154]:
lasso_data = coefficients_df[abs(coefficients_df['Coefficient']) > 0]

temp = data.reset_index()
lasso_db = temp[temp['id'].isin(lasso_data['Feature'])]
lasso_seires = extract_series(lasso_db)
lasso_country = extract_contry(lasso_seires)
lasso_category = extract_category(lasso_country)

/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_14497/395400177.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['id'] = df_temp['id'].astype(str)
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_14497/395400177.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['series'] = df_temp['id'].str[:4].astype(int)


In [155]:
lasso_category.to_excel('lasso_data.xlsx')
lasso_category.groupby('Series Name').count()['id'].sort_values(ascending=False).to_csv('lasso_series.csv')
lasso_category.groupby('Country Name').count()['id'].to_csv('lasso_country.csv')